In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/pm25_complete_model_ready.parquet")

In [2]:
print("Shape:", df.shape)
print("\nColumn names:\n", df.columns)
print("\nInfo:")
print(df.info())

Shape: (635159, 36)

Column names:
 Index(['county_code', 'site_number', 'poc', 'latitude', 'longitude',
       'date_gmt', 'time_gmt', 'sample_measurement', 'qualifier', 'site_id',
       'site_name', 'datetime', 'pm_filled', 'was_imputed', 'qc_weight',
       'time_utc', 'geometry', 'timestamp', 'hour_bucket', 'smoke_id_1',
       'smoke_id_2', 'smoke_id_3', 'smoke_dist_1', 'smoke_dist_2',
       'smoke_dist_3', 'fire_id_1', 'fire_id_2', 'fire_id_3', 'fire_dist_1',
       'fire_dist_2', 'fire_dist_3', 'met_site_id', 'met_station_dist_km',
       'site_id_met', 'wind_dir', 'wind_speed'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 635159 entries, 0 to 635158
Data columns (total 36 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   county_code          635159 non-null  int64         
 1   site_number          635159 non-null  int64         
 2   poc                  635159

In [3]:
df.head()

,county_code,site_number,poc,latitude,longitude,date_gmt,time_gmt,sample_measurement,qualifier,site_id,...,fire_id_2,fire_id_3,fire_dist_1,fire_dist_2,fire_dist_3,met_site_id,met_station_dist_km,site_id_met,wind_dir,wind_speed
0,1,13,3,37.864767,-122.302741,2020-01-01,08:00,24.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,129.3,2.8
1,1,13,3,37.864767,-122.302741,2020-01-01,09:00,27.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,142.5,4.0
2,1,13,3,37.864767,-122.302741,2020-01-01,10:00,17.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,84.1,2.0
3,1,13,3,37.864767,-122.302741,2020-01-01,11:00,14.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,190.0,2.7
4,1,13,3,37.864767,-122.302741,2020-01-01,12:00,12.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,281.0,1.9


In [4]:
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))


Missing values per column:
smoke_id_3             571580
smoke_dist_3           571580
smoke_dist_2           546979
smoke_id_2             546979
qualifier              539050
smoke_id_1             512476
smoke_dist_1           512476
fire_dist_3            217756
fire_id_3              217756
fire_id_2              156560
fire_dist_2            156560
fire_id_1               84893
fire_dist_1             84893
site_id_met             44783
sample_measurement      34100
pm_filled               31776
met_site_id                 0
met_station_dist_km         0
wind_dir                    0
county_code                 0
hour_bucket                 0
site_number                 0
timestamp                   0
geometry                    0
time_utc                    0
qc_weight                   0
was_imputed                 0
datetime                    0
site_name                   0
site_id                     0
time_gmt                    0
date_gmt                    0
longitude   

In [5]:
print("\nNumber of unique sites:", df["site_id"].nunique())

df["datetime"] = pd.to_datetime(df["datetime"])
print("\nDate range:", df["datetime"].min(), "→", df["datetime"].max())


Number of unique sites: 16

Date range: 2020-01-01 08:00:00 → 2025-01-01 07:00:00


# Modelling - ARIMA

In [2]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm


In [3]:
target_col = "pm_filled"

df = df.sort_values(["site_id", "datetime"]).reset_index(drop=True)

df_1 = df[["site_id", "datetime", "pm_filled"]].copy()
df_1["datetime"] = pd.to_datetime(df_1["datetime"])
sites = df_1["site_id"].unique()

In [4]:
def fit_arima_for_site(df_site):
    # Drop missing
    df_site = df_site.dropna(subset=["pm_filled"]).copy()
    
    # Must sort (safety)
    df_site = df_site.sort_values("datetime")

    # Need enough data
    if len(df_site) < 400:
        return None

    # -----------------------------------
    # ❗ USE INTEGER INDEX instead of datetime
    # -----------------------------------
    ts = df_site["pm_filled"].reset_index(drop=True)

    # Train/test split
    split_idx = int(len(ts) * 0.8)
    train = ts.iloc[:split_idx]
    test  = ts.iloc[split_idx:]

    if len(test) == 0:
        return None

    model = SARIMAX(
        train,
        order=(2,1,2),
        seasonal_order=(0,0,0,0),  # pure ARIMA
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    try:
        fitted = model.fit(disp=False)
    except Exception as e:
        print("Model failed for site:", df_site["site_id"].iloc[0], e)
        return None

    # Forecast
    forecast = fitted.get_forecast(steps=len(test))
    preds = forecast.predicted_mean

    # Drop nan
    valid = (~test.isna()) & (~preds.isna())
    test  = test[valid]
    preds = preds[valid]

    if len(test) == 0:
        return None

    mae  = mean_absolute_error(test, preds)
    rmse = np.sqrt(mean_squared_error(test, preds))

    return {
        "train_n": len(train),
        "test_n":  len(test),
        "mae": mae,
        "rmse": rmse,
    }


In [5]:
arima_results = []

for site in tqdm(sites):
    df_site = df_1[df_1["site_id"] == site].copy()
    res = fit_arima_for_site(df_site)
    if res is not None:
        arima_results.append({"site_id": site, **res})

arima_df = pd.DataFrame(arima_results)
print(arima_df)

 31%|███▏      | 5/16 [00:13<00:29,  2.70s/it]/opt/anaconda3/envs/safe_for_everything/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
 38%|███▊      | 6/16 [00:18<00:32,  3.23s/it]/opt/anaconda3/envs/safe_for_everything/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
100%|██████████| 16/16 [00:50<00:00,  3.18s/it]

          site_id  train_n  test_n       mae       rmse
0   06-001-0013-3    20771    5193  8.217200  10.129901
1   06-007-0008-3    33885    8472  4.890097  10.472001
2   06-013-0002-3    33612    8404  5.065066   6.159123
3   06-019-0011-3    34038    8510  8.398971  10.368446
4   06-029-0010-3    19778    4945  9.325220  16.432598
5   06-037-1103-3    20208    5052  5.905219  11.093702
6   06-037-4008-3    30896    7724  5.566341   8.628736
7   06-039-2010-3    34651    8663  6.917187   8.829244
8   06-059-0007-3    33601    8401  5.000556   8.439844
9   06-061-0003-1    33776    8444  4.626903   5.699467
10  06-065-8001-3    20573    5144  6.934390  11.497741
11  06-075-0005-3    33619    8405  5.771964   6.889943
12  06-077-2010-3    33589    8398  4.935942   8.091178
13  06-085-0002-3    31914    7979  3.215813   5.026980
14  06-099-0006-3    34081    8521  7.471557   9.545438
15  06-103-0007-3    33708    8428  5.902005   8.336808


# Modelling - SARIMA

In [6]:
def fit_sarima_for_site(df_site):
    # Drop missing PM values
    df_site = df_site.dropna(subset=["pm_filled"]).copy()
    
    # Must sort
    df_site = df_site.sort_values("datetime")

    # Need enough data for seasonal model (1+ years recommended)
    if len(df_site) < 800:  
        # seasonal models need more data
        return None

    # -----------------------------------
    # Use integer index (safe + robust)
    # -----------------------------------
    ts = df_site["pm_filled"].reset_index(drop=True)

    # Train/test split
    split_idx = int(len(ts) * 0.8)
    train = ts.iloc[:split_idx]
    test  = ts.iloc[split_idx:]

    # If test is empty (rare cleanup), skip this site
    if len(test) == 0:
        return None

    # -----------------------------------
    # SARIMA: Seasonality = 24 hours
    # -----------------------------------
    model = SARIMAX(
        train,
        order=(2,1,2),           # same ARIMA structure as baseline
        seasonal_order=(1,0,1,24),  # ADD DAILY SEASONALITY
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    try:
        fitted = model.fit(disp=False)
    except Exception as e:
        print("SARIMA failed for site:", df_site["site_id"].iloc[0], e)
        return None

    # Forecast
    forecast = fitted.get_forecast(steps=len(test))
    preds = forecast.predicted_mean

    # Remove NaNs
    valid = (~test.isna()) & (~preds.isna())
    test  = test[valid]
    preds = preds[valid]

    # skip if all invalid
    if len(test) == 0:
        return None

    # Metrics
    mae  = mean_absolute_error(test, preds)
    rmse = np.sqrt(mean_squared_error(test, preds))

    return {
        "train_n": len(train),
        "test_n":  len(test),
        "mae": mae,
        "rmse": rmse,
    }


In [7]:
sarima_results = []

for site in tqdm(sites):
    df_site = df_1[df_1["site_id"] == site].copy()
    res = fit_sarima_for_site(df_site)
    if res is not None:
        sarima_results.append({"site_id": site, **res})

sarima_df = pd.DataFrame(sarima_results)
print(sarima_df)


 19%|█▉        | 3/16 [01:33<07:13, 33.33s/it]/opt/anaconda3/envs/safe_for_everything/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
 25%|██▌       | 4/16 [02:34<08:49, 44.13s/it]/opt/anaconda3/envs/safe_for_everything/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
 31%|███▏      | 5/16 [03:30<08:52, 48.37s/it]/opt/anaconda3/envs/safe_for_everything/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
 62%|██████▎   | 10/16 [06:58<04:19, 43.32s/it]/opt/anaconda3/envs/safe_for_everything/lib/python3.10/site-pa

          site_id  train_n  test_n        mae       rmse
0   06-001-0013-3    20771    5193  10.929083  12.454110
1   06-007-0008-3    33885    8472   4.959077  10.637062
2   06-013-0002-3    33612    8404   5.216945   6.292280
3   06-019-0011-3    34038    8510   8.315292  10.300806
4   06-029-0010-3    19778    4945   9.112847  16.167030
5   06-037-1103-3    20208    5052   5.995313  11.326008
6   06-037-4008-3    30896    7724   6.643112   8.558991
7   06-039-2010-3    34651    8663   5.748117   8.151817
8   06-059-0007-3    33601    8401   5.256593   8.848653
9   06-061-0003-1    33776    8444   4.646739   5.718169
10  06-065-8001-3    20573    5144   7.167998  11.808904
11  06-075-0005-3    33619    8405   7.181201   8.234689
12  06-077-2010-3    33589    8398   5.297656   8.531125
13  06-085-0002-3    31914    7979   3.080951   4.763985
14  06-099-0006-3    34081    8521   8.682755  10.460677
15  06-103-0007-3    33708    8428   5.922086   8.350565


In [9]:
# Auto-ARIMA, automatic select better parameter set

from pmdarima import auto_arima

def fit_auto_sarima_for_site(df_site):
    # Drop missing PM values
    df_site = df_site.dropna(subset=["pm_filled"]).copy()
    df_site = df_site.sort_values("datetime")

    if len(df_site) < 800:
        return None

    # Integer index TS
    ts = df_site["pm_filled"].reset_index(drop=True)

    # Split
    split_idx = int(len(ts) * 0.8)
    train = ts.iloc[:split_idx]
    test  = ts.iloc[split_idx:]

    if len(test) == 0:
        return None

    # -------------------------------
    # Auto-ARIMA: find best p,d,q,P,D,Q
    # -------------------------------
    try:
        model = auto_arima(
            train,
            seasonal=True,
            m=24,                # daily seasonality for hourly data
            trace=False,         # set True to print search results
            error_action="ignore",
            suppress_warnings=True,
            stepwise=True        # fast heuristic search
        )
    except Exception as e:
        print("Auto-ARIMA failed for site:", df_site["site_id"].iloc[0], e)
        return None

    # -------------------------------
    # Forecast
    # -------------------------------
    preds = model.predict(n_periods=len(test))

    # Clean NaNs from preds
    preds = pd.Series(preds)
    valid = (~test.isna()) & (~preds.isna())
    test  = test[valid]
    preds = preds[valid]

    if len(test) == 0:
        return None

    mae  = mean_absolute_error(test, preds)
    rmse = np.sqrt(mean_squared_error(test, preds))

    # Extract best model orders
    order = model.order
    seasonal_order = model.seasonal_order

    return {
        "train_n": len(train),
        "test_n": len(test),
        "mae": mae,
        "rmse": rmse,
        "order": order,
        "seasonal_order": seasonal_order,
    }


In [10]:
auto_results = []

for site in tqdm(sites):
    df_site = df_1[df_1["site_id"] == site].copy()
    res = fit_auto_sarima_for_site(df_site)
    if res is not None:
        auto_results.append({"site_id": site, **res})

auto_df = pd.DataFrame(auto_results)
print(auto_df)


  0%|          | 0/16 [00:00<?, ?it/s]

: 